# A3 — Data for the option-market tests (Tests 1–3)

Builds the inputs for the three supplementary tests in **B4**, which ask
whether retail investors' pre-announcement option trading is informed, what
the options they trade actually earn across the announcement, and why.

These inputs span **both** samples, so they live in their own notebook rather
than in A1 or A2:

| Input | Coverage | Source |
|---|---|---|
| CRSP daily returns | 2010–2025 | WRDS (`crsp.dsf_v2`) |
| CRSP market index (validation only) | 2010–2024 | WRDS (`crsp.dsi`) |
| Constructed value-weighted market return | 2010–2025 | built from the two above + local CRSP caps |
| Signed daily option flow (retail, professional) | base and extension | built from `cboe_parquet(_ext)` |
| OptionMetrics option and underlying prices | 2016 – Aug 2025 | WRDS (IvyDB US, `optionm_all`) |
| Contract-level option panel (Tests 2–3) | 2016 – Aug 2025 | built from all of the above |

**Prerequisites.** A1 and A2 must have been run (this notebook reads the CBOE
parquet, CRSP prices and event panels for both samples). Steps 1, 2, 3 and 6
need a WRDS account with access to CRSP and OptionMetrics IvyDB US, and the
`wrds` Python package.

**Run once.** Every step skips work that is already on disk, and the
OptionMetrics pull resumes from its last saved chunk, so re-running is safe.
Expect roughly two hours in total, dominated by the OptionMetrics pull.

## Setup

In [ ]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl
from paths import DATA_DIR, IBES_DIR, CRSP_DIR

## Data notes (documentary)

Recorded so a reader knows why the inputs look the way they do:

- **CRSP's legacy market index ends at 2024-12-31.** `crsp.dsi` stopped
  updating when CRSP retired its legacy format, but the post-COVID era runs
  through 2025. Rather than splice a different series in for 2025, a single
  value-weighted market return is constructed for every year (Step 4) and
  validated against `vwretd` wherever `vwretd` exists.
- **OptionMetrics coverage ends at 2025-08-29** in the current annual update,
  so Tests 2 and 3 run to late August 2025 while Test 1 runs to December 2025.
  When a later update arrives, raise `date_to` in `run_optionmetrics_pull` —
  and delete the `opprcd_<year>` / `secprd_<year>` folders for affected years
  first, since saved chunks are numbered within each year.
- **IBES announcement times are not zero-padded** (`7:00:00` as well as
  `16:05:00`). `informed_trading` parses them numerically; comparing them as
  text misclassifies every before-open announcement as after-close.
- **CBOE has no data from 2022-05-17 to 2022-07-31.** Events whose flow window
  touches that gap are dropped in Test 1 rather than read as zero flow.

## Step 1 — Connect to WRDS

Prompts for a WRDS username, password and Duo approval the first time, and
offers to save a `.pgpass` file so later connections do not prompt for the
password.

In [ ]:
import wrds
db = wrds.Connection()

## Step 2 — CRSP daily returns

Total daily returns (`DlyRet`, including distributions) and returns excluding
distributions (`DlyRetx`) for every security, 2010–2025, from the CIZ-format
daily stock file. One query per year; years already on disk are skipped.
Written to `data/crsp/returns/` — a subfolder, so the CRSP price ingestion
(which reads every CSV in `data/crsp/`) cannot pick it up.

In [ ]:
RET_DIR = CRSP_DIR / "returns"
RET_DIR.mkdir(parents=True, exist_ok=True)

for year in range(2010, 2026):
    f = RET_DIR / f"crsp_returns_{year}.parquet"
    if f.exists():
        continue
    df = db.raw_sql(f"""
        SELECT permno, dlycaldt, dlyret, dlyretx
        FROM crsp.dsf_v2
        WHERE dlycaldt BETWEEN '{year}-01-01' AND '{year}-12-31'
    """, date_cols=["dlycaldt"])
    (pl.from_pandas(df)
       .rename({"permno": "PERMNO", "dlycaldt": "DlyCalDt", "dlyret": "DlyRet", "dlyretx": "DlyRetx"})
       .with_columns(pl.col("PERMNO").cast(pl.Int64), pl.col("DlyCalDt").cast(pl.Date))
       .write_parquet(f, compression="zstd"))
    print(f"{year}: {len(df):,} rows")

print("return files on disk:", len(list(RET_DIR.glob("crsp_returns_*.parquet"))), "(expect 16)")

## Step 3 — CRSP market index (for validation)

The legacy daily index file. Used only to validate the constructed market
return in Step 4; it ends at 2024-12-31.

In [ ]:
mkt_file = RET_DIR / "crsp_market_daily.parquet"
if not mkt_file.exists():
    dsi = db.raw_sql("""
        SELECT date, vwretd, vwretx, ewretd, sprtrn
        FROM crsp.dsi
        WHERE date >= '2010-01-01'
    """, date_cols=["date"])
    (pl.from_pandas(dsi).rename({"date": "DlyCalDt"})
       .with_columns(pl.col("DlyCalDt").cast(pl.Date))
       .write_parquet(mkt_file, compression="zstd"))
m = pl.read_parquet(mkt_file)
print(f"CRSP index: {m.height:,} days, {m['DlyCalDt'].min()} to {m['DlyCalDt'].max()}")

## Step 4 — Constructed value-weighted market return

Each day's market return is the average of stock returns weighted by each
stock's previous-day market cap, built for every year 2010–2025. Two versions
are built (all securities; common stocks only) and both are compared with
CRSP's `vwretd` over 2010–2024.

**Expected:** the all-securities series matches `vwretd` with a correlation of
about 0.9998 and an average gap of about 1.5 bp a day, with no year below
0.999. It is the series Test 1 uses (`mkt_vw_all`).

In [ ]:
from pipeline.build_market_return import build_market_return, validate_market_return, OUT_PATH

mkt_vw = build_market_return() if not OUT_PATH.exists() else pl.read_parquet(OUT_PATH)
val = validate_market_return(mkt_vw)
with pl.Config(tbl_rows=-1, float_precision=4):
    print(val)

## Step 5 — Signed daily option flow

Retail and professional contract volume per ticker-day, split by call/put ×
opening/closing × buy/sell and summed across size tiers (16 columns). The
existing daily retail table collapses the buy/sell side away; this table keeps
it without touching that table, which the verified thesis results depend on.

Watch each year's line for an "unrecognised call/put flag" note: a handful of
rows is harmless, a large count would mean a coding change in the CBOE export.

In [ ]:
from pipeline.build_daily_flow import build_daily_flow

for src, out in [("cboe_parquet", "cboe_daily_flow"), ("cboe_parquet_ext", "cboe_daily_flow_ext")]:
    if not list((DATA_DIR / out).glob("daily_flow_*.parquet")):
        build_daily_flow(cboe_dir=DATA_DIR / src, out_dir=DATA_DIR / out)
    else:
        print(f"{out} already exists -- skipping.")

### Verify: both flow tables present, 16 flow columns

In [ ]:
from pipeline.build_daily_flow import flow_columns

for out, n_expected in [("cboe_daily_flow", 12), ("cboe_daily_flow_ext", 5)]:
    files = sorted((DATA_DIR / out).glob("daily_flow_*.parquet"))
    cols = pl.scan_parquet(files[0]).collect_schema().names()
    missing = set(flow_columns()) - set(cols)
    print(f"{out}: {len(files)} files (expect {n_expected}); missing flow columns: {missing or 'none'}")

## Step 6 — OptionMetrics extract

Option prices, implied volatilities and Greeks (`opprcd`) and underlying
prices (`secprd`) for trading days −1 to +2 around every earnings
announcement from 2016 to 27 August 2025, restricted to contracts expiring
within 120 days. Events link to OptionMetrics by 8-digit CUSIP, matched as of
the announcement date.

About 535,000 (firm, date) pairs in about 270 chunks; roughly 1.5–2 hours.
**Resumable:** every chunk is saved as it arrives, so after a dropped
connection, reconnect (Step 1) and run this cell again. Keep the machine awake
while it runs.

In [ ]:
from pipeline.pull_optionmetrics import run_optionmetrics_pull

run_optionmetrics_pull(db)

### Verify: every chunk saved, and the IvyDB conventions hold

Chunk completeness is checked against the number of chunks each year's pairs
require. The convention checks confirm what the Test 3 decomposition relies
on: implied volatility stored as a decimal (median around 0.5), no negative
vega placeholders, and nearly all contracts standard-settlement
(`ss_flag = '0'`).

In [ ]:
import math
from pipeline.pull_optionmetrics import OM_DIR, CHUNK_SIZE

pairs = pl.read_parquet(OM_DIR / "event_pairs.parquet")
need = pairs.select("secid", "date", "year").unique().group_by("year").len().sort("year")
for year, n_pairs in need.iter_rows():
    expected = math.ceil(n_pairs / CHUNK_SIZE)
    n_opt = len(list((OM_DIR / f"opprcd_{year}").glob("part_*.parquet")))
    n_sec = len(list((OM_DIR / f"secprd_{year}").glob("part_*.parquet")))
    flag = "" if n_opt == n_sec == expected else "   <-- MISSING CHUNKS: re-run Step 6"
    print(f"{year}: {n_opt}/{expected} option parts, {n_sec}/{expected} security parts{flag}")

opt = pl.scan_parquet(OM_DIR / "opprcd_2016" / "*.parquet")
print(opt.select(
    pl.col("impl_volatility").median().alias("iv_median"),
    (pl.col("vega") < 0).sum().alias("vega_negative"),
    (pl.col("ss_flag") == "0").mean().alias("share_standard"),
).collect())

## Step 7 — Contract-level option panel (Tests 2 and 3)

Builds the Test 1 event panel (for the timing-corrected day 0), then matches
every contract retail and professional customers traded on day −1 to its
OptionMetrics quotes at day −1 and day +1, with bid, ask, IV and Greeks at
both ends. Saved to `data/optionmetrics/option_contract_panel.parquet` so B4
reads it directly. Takes several minutes.

**Expected:** the timing split is realistic (roughly 43% before-open, 55%
after-close), about 89% of retail and 84% of professional day −1 buy volume is
priced, and about 139,000 contract-events are valued at their payoff because
they expired inside the window.

In [ ]:
from analysis.informed_trading import build_event_panel
from analysis.option_returns import build_option_panel, PANEL_PATH

panel = build_event_panel()
stale = PANEL_PATH.exists() and "bid_entry" not in pl.scan_parquet(PANEL_PATH).collect_schema().names()
if not PANEL_PATH.exists() or stale:
    if stale:
        print("Saved contract panel predates the bid/ask columns -- rebuilding.")
    build_option_panel(panel)
else:
    print("option_contract_panel.parquet already exists -- skipping.")

## Option-market data built

| Input | Location |
|---|---|
| CRSP daily returns | `crsp/returns/crsp_returns_<year>.parquet` |
| CRSP market index (validation) | `crsp/returns/crsp_market_daily.parquet` |
| Constructed market return | `crsp/returns/market_vw_constructed.parquet` |
| Signed daily flow | `cboe_daily_flow/`, `cboe_daily_flow_ext/` |
| OptionMetrics extract | `optionmetrics/opprcd_<year>/`, `secprd_<year>/`, `secnmd.parquet`, `event_pairs.parquet` |
| Contract-level option panel | `optionmetrics/option_contract_panel.parquet` |

The three tests are in **B4**.